<!-- SPDX-License-Identifier: Apache-2.0 -->
<!-- Copyright (c) 2025 FlyDSL Project Contributors -->

# Numeric types

Every scalar you compute with in a FlyDSL kernel has a **DSL numeric type** —
`fx.Int32`, `fx.Float32`, `fx.BFloat16`, and so on. The type fixes a value's
precision and signedness, decides how arithmetic and casts behave, and tells the
compiler what to emit.

This notebook is about *using* these types: the families to pick from, how to
construct and compute with values, casts and promotion, and the difference between
**compile-time** (`Constexpr`) and **runtime** values. How a DSL type maps onto an
MLIR type underneath is implementation — it lives in an optional section at the end
(§6), so skip it if you just want to write kernels.

In [ ]:
import os

# §6 prints at trace time; disable the JIT disk cache so a warm-cache re-run still
# re-traces and re-prints (these kernels are tiny). Set before importing flydsl.
os.environ["FLYDSL_RUNTIME_ENABLE_CACHE"] = "0"

import torch
from wurlitzer import pipes

import flydsl.compiler as flyc
import flydsl.expr as fx


def show_gpu_output(launcher, *args, **kwargs):
    """Run a @flyc.jit launcher and echo its GPU printf output into the notebook.
    (Device printf isn't captured by Jupyter on its own; wurlitzer routes it here.)"""
    kwargs.setdefault("stream", torch.cuda.Stream())
    with pipes() as (out, _err):
        launcher(*args, **kwargs)
        torch.cuda.synchronize()
    print(out.read(), end="")

## 1. The type families

| Family | DSL types | When you reach for it |
|---|---|---|
| Signed int | `Int4`, `Int8`, `Int16`, `Int32`, `Int64` | indices, counters, masks |
| Unsigned int | `Uint8`, `Uint16`, `Uint32`, `Uint64` | bit math, addresses, packed lanes |
| Float | `Float16`, `BFloat16`, `Float32`, `Float64` | math and accumulators; `Float32` is the default |
| FP8 / FP6 / FP4 | `Float8E5M2`, `Float8E4M3FN`, …, `Float6E2M3FN`, `Float4E2M1FN` | low-precision matmul / copy operands on CDNA |
| Special | `Boolean`, `Index` | predicates; loop and index arithmetic |

Pick by precision and signedness, the same way you would choose a C or `torch`
dtype. (The exact MLIR type each one lowers to is in §6 — you don't need it yet.)

Each type exposes its bit `.width`, and this is pure metadata, so it reads at the
notebook top level with no GPU and no kernel:

In [ ]:
for t in [
    fx.Int8,
    fx.Int32,
    fx.Uint32,
    fx.Float16,
    fx.BFloat16,
    fx.Float32,
    fx.Float8E4M3FN,
    fx.Float4E2M1FN,
    fx.Boolean,
    fx.Index,
]:
    print(f"{t.__name__:<14} width = {t.width}")

## 2. Constructing and computing

A value is created by calling the type: `fx.Int32(7)`, `fx.Float32(3.5)`.

Constructing and operating on values only works *inside a traced kernel* — there is
no compiler context at the notebook top level, so this all happens under
`@flyc.kernel` / `@flyc.jit`. We surface the results with `fx.printf`.

The usual operators work: `+ - * // / %`, bitwise `& | ^ ~ << >>`, and comparisons
`< <= > >= == !=` (which return a `Boolean`). Two `printf` quirks: avoid a literal
`%` in the format string (the device `printf` consumes it), and a true `Boolean`
prints as `-1` (all bits set).

In [ ]:
@flyc.kernel
def numeric_demo():
    a = fx.Int32(7)
    b = fx.Int32(3)
    fx.printf("a+b={}  a-b={}  a*b={}  a//b={}  a-mod-b={}", a + b, a - b, a * b, a // b, a % b)
    fx.printf("a&b={}  a|b={}  a^b={}  a<<1={}", a & b, a | b, a ^ b, a << fx.Int32(1))
    fx.printf("a>b -> {} (true is -1)   a==b -> {}", a > b, a == b)


@flyc.jit
def run_numeric(stream: fx.Stream = fx.Stream(None)):
    numeric_demo().launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_numeric)

## 3. Casts and type promotion

Convert explicitly with `.to(TargetType)`. In a mixed-type expression FlyDSL also
**promotes** operands to a common type (wider wins; float beats int; on a tie,
unsigned beats signed). The `fx.math` module provides the usual float functions
(`sqrt`, `exp`, `exp2`, `log`, `sin`, `fma`, …).

In [ ]:
@flyc.kernel
def cast_demo():
    i = fx.Int32(7)
    f = i.to(fx.Float32)  # explicit int -> float
    g = fx.Float32(2.0)
    fx.printf("f/g={}  sqrt(f)={}  exp2(g)={}", f / g, fx.math.sqrt(f), fx.math.exp2(g))

    # mixed: Int32 + Float32 promotes to Float32
    promoted = fx.Int32(3) + g
    fx.printf("int(3) + float(2.0) = {}  (promoted to f32)", promoted)


@flyc.jit
def run_cast(stream: fx.Stream = fx.Stream(None)):
    cast_demo().launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_cast)

## 4. Compile-time (`Constexpr`) vs runtime values

A kernel parameter typed as `fx.Constexpr[T]` is a **Python value at trace time** —
it can size loops, pick code paths, and is baked into the compiled kernel (it is
part of the cache key, so a new value triggers a recompile). A parameter typed as a
numeric type (e.g. `fx.Int32`) is a **runtime IR value**, materialized in the kernel.

In [ ]:
@flyc.kernel
def constexpr_demo(scale: fx.Constexpr[int], x: fx.Int32):
    # `scale` is a real Python int here — we can use it in plain Python.
    label = "doubling" if scale == 2 else f"x{scale}"
    fx.printf("scale is a python int at trace time: " + label)
    fx.printf("x * scale = {}", x * scale)


@flyc.jit
def run_constexpr(x: fx.Int32, scale: fx.Constexpr[int], stream: fx.Stream = fx.Stream(None)):
    constexpr_demo(scale, x).launch(grid=(1, 1, 1), block=(1, 1, 1), stream=stream)


show_gpu_output(run_constexpr, fx.Int32(10), 4)

## 5. Low-precision types

AMD GPUs lean heavily on reduced precision, so the half- and sub-8-bit formats in
the table above (`BFloat16`, the `Float8*` / `Float6*` / `Float4*` families) are
first-class DSL types — you declare buffers and accumulators with them just like
`Float32`. Their narrow mantissas trade accuracy for speed and memory bandwidth.
The payoff shows up where data is *moved* and *multiplied* in low precision — the
copy atoms (next notebook) and the MMA atoms (the later layout / MMA series) —
rather than in scalar arithmetic, so we only introduce the type names here.

## 6. Under the hood — DSL type → MLIR (optional)

Skip this if you only want to write kernels — you will rarely reach for it. But the
moment you dump IR (`00_hello_flydsl`) or read a type error, you hit MLIR type names
like `i32` / `f32` / `f8E4M3FN`, and this section is the map back to the DSL types you
wrote. Under the hood each DSL type is a thin handle over one **MLIR scalar type**: it
knows that type and emits the right `arith` / `math` ops for it.

| DSL family | MLIR type |
|---|---|
| Signed int | `i4 … i64` |
| Unsigned int | `i8 … i64` (same signless `iN` — see below) |
| Float | `f16`, `bf16`, `f32`, `f64` |
| FP8 / FP6 / FP4 | `f8E4M3FN`, `f8E5M2`, …, `f6E2M3FN`, `f4E2M1FN` (OCP) |
| Special | `i1` (Boolean), `index` (Index) |

The mapping is observable through a type's `.ir_type` — but only *inside* a trace,
because building an MLIR type needs a compiler context (the same reason arithmetic
had to live in a kernel back in §2). Notice the `print`s below run at *trace* time,
so their output lands here directly — no GPU, no `wurlitzer`.

Watch `Uint32` and `Int32`: both report `i32`. MLIR integers are **signless** — the
type carries the bit width but no sign bit. Signedness rides on the *operation* instead
(signed divide is `arith.divsi`, unsigned is `arith.divui`), which is why `Int32` and
`Uint32` stay distinct DSL types even though they share one MLIR type — and why the §1
table never spelled out a `uN` type: there isn't one.

In [ ]:
@flyc.jit
def show_ir_types():
    for t in (fx.Int32, fx.Uint32, fx.Float32, fx.BFloat16, fx.Float8E4M3FN, fx.Boolean, fx.Index):
        print(f"{t.__name__:<14} width={t.width:<3} -> MLIR {t.ir_type}")
    # types compose: four f32 packed into one vector value
    print("Vector.make_type(4, Float32) ->", fx.Vector.make_type(4, fx.Float32))


show_ir_types()

That `vector<4xf32>` is how a thread holds four `f32` values at once — the packing the
copy and layout notebooks lean on when a thread moves or multiplies several elements
together.

---
**Next:** [`02_struct`](02_struct.ipynb) — bundling these scalars into aggregate
value types with `@fx.struct`.